[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [3]:
test_tensor = torch.rand((2,3))
padded = F.pad(test_tensor, (2,2,2,2))
padded

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.8017, 0.3941, 0.1481, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.9142, 0.8998, 0.6881, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])

In [24]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    x = F.pad(x, (padding, padding, padding, padding))
    patch_H = weight.shape[-2] # row
    patch_W = weight.shape[-1] # col
    total_W = x.shape[-1]
    total_H = x.shape[-2]
    W_idx = 0
    new_W = []
    while W_idx + patch_W <= total_W:
        new_H = []
        H_idx = 0
        while H_idx + patch_H <= total_H:
            patch = x[:, :, H_idx:H_idx+patch_H, W_idx:W_idx+patch_W] # (B, C_in, kH, kW)
            # apply kernel
            output = torch.einsum("bchw,kchw->bk", patch, weight)
            new_H.append(output)
            H_idx += stride

        # print(new_H)
        # stack from N (B, C_out) to (B, C_out, N)
        new_H_stack = torch.stack(new_H, dim=-1)
        new_W.append(new_H_stack)
        W_idx += stride

    # M (B, C_out, N) to (B, C_out, N, M)
    new_W_stack = torch.stack(new_W, dim=-1)
    if bias is not None:
      return new_W_stack + bias.reshape(1, -1, 1, 1) # NOTE: bias shape is (C_out,)
    else:
      return new_W_stack


# TODO: implement via unfold

In [25]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print(f"Expected: {F.conv2d(x, w).shape}")
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Expected: torch.Size([1, 16, 6, 6])
Match: True


In [26]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.9ms)
  ✅ [2/5] Matches F.conv2d (8.1ms)
  ✅ [3/5] With padding (6.3ms)
  ✅ [4/5] With stride (3.7ms)
  ✅ [5/5] Gradient flow (2.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (25.2ms total)
  Progress saved. Run status() to see your dashboard.

